# Context-management cost analysis -- sprint 4 (bd polyglot-pdf-6x1y)

Interactive analysis of the bd-`wyhh` three-arm matrix: **2 models x {unbounded,
section-reset, sliding-window} x {happy, unhappy} x 20 questions** against live
Gemini, driven through the real production stack (bd `4loz`).

**Trust gate:** every computation in this notebook comes from
`cost_matrix_lib.py`, which passed a 3-round adversarial fresh-eyes review
(round 1 caught a window-model formula error and a mispaired calibration
anchor; round 2 caught a sign-flipping span mismatch and a global-state leak;
round 3 verdict: TRUSTED). The notebook only loads, models, and renders.

**Data caveats baked into the loaders** (not footnotes -- encoded as `valid` /
`degraded` flags): the two 3.1 single-session unhappy cells were terminated by
the session-duration GoAway at ~q12-13; one 2.5 section-reset cell lost q16-17
to a `1006` transient (recovered q18-20); 2.5 cells carry occasional silent
`update_form_fields` TypeErrors (bd `d24n`).

In [1]:
import sys, json
from pathlib import Path
# Location-independent imports: works with cwd at the repo root OR at
# scripts/spikes (jupyter execute uses the notebook's own directory).
for cand in (Path.cwd(), Path.cwd() / 'scripts' / 'spikes'):
    if (cand / 'cost_matrix_lib.py').exists():
        sys.path.insert(0, str(cand))
        break
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cost_matrix_lib as L

questions, turns, curves, runs = L.load_matrix()     # validates + calibrates
print(f"loaded {len(questions)} question rows / {len(turns)} turns / 12 runs")
print(f"calibration: {L.CHAR_RATE:.4f} tok/char + {L.FIXED_OVERHEAD:.0f} tok overhead")
live = L.recompute_si_scenarios()
drift = {k: live[s] - v for (k, v), s in zip(L.SI_SCENARIOS_CHARS.items(), ('full','jit','drip'))}
print('SI literal drift vs live recompute (chars):', drift, '+ sections:',
      [a-b for a, b in zip(live['sections'], L.SECTION_SI_CHARS)])

loaded 240 question rows / 720 turns / 12 runs
calibration: 0.2798 tok/char + 992 tok overhead
2026-06-12 18:09:43

 [

info     

] 

manifest_generated            

...

0

SI literal drift vs live recompute (chars):

{"full-dump (today's no-sections port)": 0, 'jit-manifest (built, unmeasured live)': 0, 'drip-in (fields=[]; schema via tools)': 0}

+ sections:

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

## 1. Data integrity panel

Run-level caveats first -- every later chart excludes invalid (dead-session)
questions unless explicitly labeled otherwise.

In [2]:
summary = L.arm_summary(questions, runs)
display(summary[['model','arm','path','questions_counted','total_tokens','total_cost',
                 'text_in_share','confirmed_fields','n_stalled_turns','truncated']]
        .style.format({'total_cost': '${:.3f}', 'text_in_share': '{:.1%}'}))
print('truncated = session killed mid-run (GoAway / 1006); questions_counted is the valid basis')

,model,arm,path,questions_counted,total_tokens,total_cost,text_in_share,confirmed_fields,n_stalled_turns,truncated
0,2.5,section-reset,happy,20,213867,$0.188,95.0%,16,0,False
1,2.5,sliding-window,happy,20,534428,$0.330,98.0%,15,0,False
2,2.5,unbounded,happy,20,535613,$0.338,97.9%,13,0,False
3,2.5,section-reset,unhappy,18,444425,$0.323,96.4%,15,11,True
4,2.5,sliding-window,unhappy,20,1342518,$0.797,97.9%,19,0,False
5,2.5,unbounded,unhappy,20,1317634,$0.781,98.0%,18,0,False
6,3.1,section-reset,happy,20,341266,$0.627,58.1%,19,0,False
7,3.1,sliding-window,happy,20,828400,$1.023,81.2%,20,0,False
8,3.1,unbounded,happy,20,1047190,$1.624,67.0%,18,0,False
9,3.1,section-reset,unhappy,20,840219,$1.729,46.9%,20,0,False


truncated = session killed mid-run (GoAway / 1006); questions_counted is the valid basis

## 2. Where the money goes -- cost sources

The four billable streams per arm (valid questions only). The 2.5 story is
one-note: **text input is 95-98% of every arm's cost** -- it is the re-billed
system instruction + accumulated conversation. 3.1 shifts the mix because its
verbose output (thinking tokens bill as output) is expensive, not because
text-in shrinks.

In [3]:
src = summary.melt(
    id_vars=['model','arm','path'],
    value_vars=['text_in_tokens','audio_in_tokens','text_out_tokens','audio_out_tokens'],
    var_name='stream', value_name='tokens')
rate_map = {('2.5','text_in_tokens'):0.50, ('2.5','audio_in_tokens'):3.00,
            ('2.5','text_out_tokens'):2.00, ('2.5','audio_out_tokens'):12.00,
            ('3.1','text_in_tokens'):0.75, ('3.1','audio_in_tokens'):3.00,
            ('3.1','text_out_tokens'):4.50, ('3.1','audio_out_tokens'):12.00}
src['cost'] = [t * rate_map[(m, s)] / 1e6 for m, s, t in zip(src['model'], src['stream'], src['tokens'])]
fig = px.bar(src, x='arm', y='cost', color='stream', facet_col='path', facet_row='model',
             title='Cost by billable stream (valid questions, 20-question runs)',
             labels={'cost': 'USD'}, height=560)
fig.show()

## 3. Cumulative cost race, question by question

The 20-question measured race per arm. Note the slopes: unbounded arms
steepen (context compounds), section-reset saw-tooths shallow (resets), the
sliding-window arms track unbounded almost exactly at this scale -- the 12k
trigger over an 8.2k unevictable instruction floor leaves almost nothing to
evict, and firing was unreliable besides (see section 6).

In [4]:
v = questions[questions['valid']].sort_values('q').copy()
v['cum_cost'] = v.groupby(['model','arm','path'])['total cost'].cumsum()
fig = px.line(v, x='q', y='cum_cost', color='arm', facet_col='path', facet_row='model',
              markers=True, height=560, labels={'cum_cost': 'cumulative USD', 'q': 'question'},
              title='Measured cumulative cost (invalid/dead questions excluded)')
fig.show()

## 4. Context length from start to question 20

Per-billed-event prompt size -- the raw "how much memory is the model carrying"
trace. Markers: red = compression eviction (sliding-window arm only),
orange = section-restart reset, gray = post-death anomaly (the GoAway cells).
This chart IS the turn-memory-vs-context-length distribution per arm.

In [5]:
fig = px.line(curves, x='event_index', y='prompt_tokens', color='arm',
              facet_col='path', facet_row='model', height=560,
              title='Prompt (context) tokens per billed event',
              labels={'event_index': 'billed event #', 'prompt_tokens': 'prompt tokens'})
drop_colors = {'eviction': 'red', 'restart': 'orange', 'anomaly': 'gray'}
drops = curves[curves['drop']]
for kind, color in drop_colors.items():
    d = drops[drops['drop_kind'] == kind]
    if len(d):
        # overlay drop markers on the matching facets
        for (m, p), dd in d.groupby(['model', 'path']):
            fig.add_trace(go.Scatter(
                x=dd['event_index'], y=dd['prompt_tokens'], mode='markers',
                marker=dict(color=color, size=9, symbol='x'),
                name=f'{kind}', legendgroup=kind, showlegend=(m == '2.5' and p == 'happy'),
            ), row=(1 if m == '3.1' else 2), col=(1 if p == 'happy' else 2))
fig.show()

## 5. Latency

Per-turn responsiveness by arm (the metric a voice product lives on). Two
verified facts: **eviction events cost nothing in latency** (medians flat
before/after first eviction; the lib's split method), and section restarts add
their own pause at boundaries (not shown here: the restart greeting wait).
2.5 vs 3.1 medians are close on this harness; the production complaint about
3.1 is its end-to-end audio-path latency, not measured here.

In [6]:
lt = turns.dropna(subset=['latency_s'])
fig = px.box(lt, x='arm', y='latency_s', color='model', facet_col='path', height=480,
             title='Turn latency: activity-end to first audio byte (s)')
fig.show()
display(L.latency_summary(turns).style.format({'median': '{:.2f}s', 'p90': '{:.2f}s'}))

,model,arm,path,n,median,p90
0,2.5,section-reset,happy,39,2.17s,3.99s
1,2.5,section-reset,unhappy,68,2.56s,3.51s
2,2.5,sliding-window,happy,39,1.75s,2.64s
3,2.5,sliding-window,unhappy,79,1.73s,2.78s
4,2.5,unbounded,happy,39,1.81s,2.90s
5,2.5,unbounded,unhappy,79,1.61s,2.62s
6,3.1,section-reset,happy,40,1.84s,2.19s
7,3.1,section-reset,unhappy,80,1.75s,2.05s
8,3.1,sliding-window,happy,40,1.88s,2.08s
9,3.1,sliding-window,unhappy,52,1.80s,1.95s


## 6. Honesty check: the window model vs what actually happened

Before any what-if chart, the model must face the measured cells. The model is
**best-case** (server evicts exactly as configured). Measured reality at
trigger=12k/window=9k: firing was unreliable (0 evictions in the 2.5 happy
cell despite crossing the trigger at ~q8), so measured costs land between the
model and the unbounded bound. At SMALL triggers (1.5k/800, minimal SI) the
h04c matrix measured **reliable** firing (3/3 on both models, 20-turn runs) --
the lean-SI scenarios below operate in that reliable regime, but that
reliability-at-scale is a stated assumption, not a measured fact.

In [7]:
growth = L.conversation_growth_per_turn(curves, runs)
display(growth.style.format({'growth_per_question': '{:.0f}'}))
check = L.window_model_check(questions, runs, growth)
display(check.style.format({c: '${:.3f}' for c in ('modeled_text_in','measured_text_in','unbounded_text_in')}))
print('Read: modeled = best-case eviction. Three rows land between modeled and unbounded')
print('(firing shortfall); the 2.5-unhappy row measured slightly ABOVE unbounded --')
print('2 late evictions bought nothing, within single-run variance but a fair warning')
print('that compression at 12k/9k over the full-dump floor can be a pure no-op.')

,model,path,growth_per_event,events_per_question,growth_per_question
0,2.5,happy,233.000000,2.000000,466
1,2.5,unhappy,218.000000,4.000000,872
2,3.1,happy,736.000000,2.000000,1472
3,3.1,unhappy,766.000000,3.833333,2936


,model,path,n_questions,modeled_text_in,measured_text_in,unbounded_text_in,measured_evictions
0,2.5,happy,20,$0.210,$0.262,$0.262,0
1,2.5,unhappy,20,$0.423,$0.657,$0.646,2
2,3.1,happy,20,$0.323,$0.504,$0.526,4
3,3.1,unhappy,12,$0.374,$0.520,$0.538,16


Read: modeled = best-case eviction. Three rows land between modeled and unbounded

(firing shortfall); the 2.5-unhappy row measured slightly ABOVE unbounded --

2 late evictions bought nothing, within single-run variance but a fair warning

that compression at 12k/9k over the full-dump floor can be a pure no-op.

## 7. The lever: what the system instruction floor does to a sliding window

Three schema-feeding scenarios, all built from the real demo schema and the
real backend templates (sizes recomputed live in cell 1):

| scenario | SI chars | unevictable floor (tokens) |
|---|---|---|
| full dump (today's no-sections port) | 25,632 | ~8,160 |
| JIT manifest (already built; never measured live) | 10,204 | ~3,850 |
| drip-in (`fields=[]`; schema arrives via tool responses, so it is EVICTABLE) | 3,069 | ~1,850 |

The maintainer's design point (2026-06-12): the giant dump is NOT an inherent
cost of windowing -- the exploration tools (`get_field_value`,
`get_next_field_schema_for_jit`) let the model *discover* field parameters,
and dripped-in schema ages out of the window naturally.

In [8]:
N_FULL_FORM = 174
schema = json.loads(L.SCHEMA_PATH.read_text(encoding='utf-8'))
sec_counts = {}
for f in schema['fields']:
    sec_counts[f.get('section_id') or 's1'] = sec_counts.get(f.get('section_id') or 's1', 0) + 1
QPS = list(sec_counts.values())
G = {(r_['model'], r_['path']): (r_['growth_per_question'], r_['events_per_question'])
     for _, r_ in growth.iterrows()}

def whole_form_curves(model, path, retained_grid=range(250, 6001, 250)):
    g, epq = G[(model, path)]
    rows = []
    for label, chars in L.SI_SCENARIOS_CHARS.items():
        base = L.si_tokens_from_chars(chars)
        for retained in retained_grid:
            window = base + retained          # keep `retained` tokens of conversation
            trigger = int(window * 1.3)
            df = L.window_cost_curve(chars, g, N_FULL_FORM, model, [(trigger, int(window))],
                                     events_per_q=epq)
            rows.append({'scenario': label, 'retained_memory_tokens': retained,
                         'text_in_cost_174q': df['cum_text_in_cost'].iloc[-1]})
    out = pd.DataFrame(rows)
    out['model'], out['path'] = model, path
    return out

wf = pd.concat([whole_form_curves('2.5', 'happy'), whole_form_curves('2.5', 'unhappy')])
g_h, epq_h = G[('2.5', 'happy')]; g_u, epq_u = G[('2.5', 'unhappy')]
sr_h = L.section_reset_model_cost(g_h, QPS, '2.5', events_per_q=epq_h)
sr_u = L.section_reset_model_cost(g_u, QPS, '2.5', events_per_q=epq_u)
fig = px.line(wf, x='retained_memory_tokens', y='text_in_cost_174q', color='scenario',
              facet_col='path', height=480, log_y=True,
              title='Whole form (174 questions, 2.5): modeled text-in cost vs retained conversation memory',
              labels={'retained_memory_tokens': 'retained conversation (tokens kept after eviction)',
                      'text_in_cost_174q': 'text-in USD (174 q, log scale)'})
for i, (sr, pth) in enumerate(((sr_h, 'happy'), (sr_u, 'unhappy')), start=1):
    fig.add_hline(y=sr, line_dash='dash', line_color='black', col=i,
                  annotation_text=f'section-reset flow: ${sr:.2f}', annotation_position='top left')
fig.show()
print(f'section-reset modeled whole-form text-in: happy ${sr_h:.2f} / unhappy ${sr_u:.2f}')
print('NOTE: best-case model; +audio/output adds ~$0.07-0.20 per 20q regardless of arm.')

section-reset modeled whole-form text-in: happy $1.61 / unhappy $5.18

NOTE: best-case model; +audio/output adds ~$0.07-0.20 per 20q regardless of arm.

### Reading the elbow

Cost is nearly **flat in retained memory** and **steps hugely with the SI
scenario** -- the floor, not the window, is the lever. There is no sharp
elbow to hunt: pick retained memory for *behavioral* reasons (enough recent
turns for coherent dialog; ~1,500-2,000 tokens is roughly the last 3-6
exchanges at measured growth rates) and spend the engineering effort on the
SI diet instead. Concretely, with an 80/20 happy/unhappy mix at 174
questions, retained memory 1,750 tokens (2.5, text-in, best-case -- the
table below is the source of record): section resets ~$2.32; full-dump
window ~$2.38 (NO better than sections); JIT window ~$1.34; drip window
~$0.86.

In [9]:
MIX = 0.8
mix_rows = []
for label, chars in L.SI_SCENARIOS_CHARS.items():
    base = L.si_tokens_from_chars(chars)
    retained = 1750
    window = int(base + retained); trigger = int(window * 1.3)
    ch = L.window_cost_curve(chars, g_h, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_h)['cum_text_in_cost'].iloc[-1]
    cu = L.window_cost_curve(chars, g_u, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_u)['cum_text_in_cost'].iloc[-1]
    mix_rows.append({'flow': f'sliding window / {label}', 'config': f'trigger {trigger:,} / window {window:,}',
                     'happy_174q': ch, 'unhappy_174q': cu, 'mix_80_20': MIX*ch + (1-MIX)*cu})
mix_rows.append({'flow': 'section-reset (current production)', 'config': '15 restarts',
                 'happy_174q': sr_h, 'unhappy_174q': sr_u, 'mix_80_20': MIX*sr_h + (1-MIX)*sr_u})
mix = pd.DataFrame(mix_rows).sort_values('mix_80_20')
display(mix.style.format({c: '${:.2f}' for c in ('happy_174q','unhappy_174q','mix_80_20')}))

,flow,config,happy_174q,unhappy_174q,mix_80_20
2,sliding window / drip-in (fields=[]; schema via tools),"trigger 4,680 / window 3,600",$0.72,$1.44,$0.86
1,"sliding window / jit-manifest (built, unmeasured live)","trigger 7,274 / window 5,596",$1.12,$2.24,$1.34
3,section-reset (current production),15 restarts,$1.61,$5.18,$2.32
0,sliding window / full-dump (today's no-sections port),"trigger 12,886 / window 9,913",$1.98,$3.96,$2.38


## 8. Interactive explorer (live kernel only)

Sliders: happy/unhappy mix and retained memory; recomputes the comparison
table. (Static defaults above are the committed record; this cell needs a
running kernel.)

In [10]:
try:
    import ipywidgets as W
    @W.interact(mix=W.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05, description='happy share'),
                retained=W.IntSlider(value=1750, min=250, max=6000, step=250, description='retained tok'))
    def explore(mix=0.8, retained=1750):
        rows = []
        for label, chars in L.SI_SCENARIOS_CHARS.items():
            base = L.si_tokens_from_chars(chars)
            window = int(base + retained); trigger = int(window * 1.3)
            ch = L.window_cost_curve(chars, g_h, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_h)['cum_text_in_cost'].iloc[-1]
            cu = L.window_cost_curve(chars, g_u, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_u)['cum_text_in_cost'].iloc[-1]
            rows.append({'flow': label, 'mix': mix*ch + (1-mix)*cu})
        rows.append({'flow': 'section-reset', 'mix': mix*sr_h + (1-mix)*sr_u})
        display(pd.DataFrame(rows).sort_values('mix').style.format({'mix': '${:.2f}'}))
except Exception as e:
    print('ipywidgets unavailable in this environment:', e)

interactive(children=(FloatSlider(value=0.8, description='happy share', max=1.0, step=0.05), IntSlider(value=1…

## 9. Cost/benefit and recommendations

### What the measured data settles
1. **Text-in is the bill.** 95-98% of 2.5 spend; the re-billed instruction +
   conversation. Audio is noise by comparison. (Confirms the maintainer's
   original cost finding with controlled data.)
2. **At today's architecture, section resets beat the naive single-session
   port at any scale** -- measured at 20 questions ($0.19 vs $0.33-0.34
   happy), modeled 1.4-5x at 174 questions vs unbounded's $8.5-29.
3. **Compression works on both models but cannot evict the instruction** --
   at 12k/9k over the 8.2k full-dump floor it recovered ~0-4% measured.
   Firing is also not guaranteed at large triggers (0 evictions in one cell).
4. **No latency cost at evictions; the duration GoAway is a separate wall**
   (~13-15 min): any single-session flow needs GoAway reconnect + resumption
   (`c5t` shipped, `l3a.2` policy open, `wnt` decided) regardless of cost.

### What the modeling (trusted code, best-case framing) indicates
5. **The schema-feeding strategy is worth 2-5x; the window size is worth
   almost nothing past ~1.5-2k retained.** A sliding window only competes
   with -- and then beats -- section resets when the SI goes on a diet:
   at 1,750 tokens retained memory, full-dump ~$2.38 (no better than
   sections' ~$2.32 -- the naive port loses at any scale), JIT-manifest
   ~$1.34 (-42%), drip ~$0.86 (-63%) per 174-question form (80/20 mix,
   text-in, 2.5).
6. **The hybrid is also viable:** fewer/larger sections WITH a window inside
   each would cut restart count and bound within-section growth (the
   54-field medical section is the section-reset flow's own pain point).

### Recommended path (in order)
1. **Fix bd `d24n`** (silent 2.5 update TypeErrors) -- it taints every
   compliance number regardless of architecture.
2. **Measure the JIT-manifest flow live** (it is built and flag-gated):
   one 20-question cell, sliding window sized base+~1,750 retained
   (trigger ~7.3k / window ~5.6k). This converts scenario 5 from model to
   measurement -- and tests eviction-firing reliability in the lean regime.
3. **Behavioral gate:** recall-after-eviction test (the model must re-fetch
   evicted facts via `get_form_status`/`get_field_value` instead of
   confabulating).
4. **Decide the arm** with that data: if JIT+window measures near the model,
   the single-session flow wins on cost AND removes 14 restart pauses of UX;
   if not, keep sections (possibly hybrid) -- they remain a strong, measured
   baseline.
5. Either way, **session lifetime work** (`l3a.2` reconnect policy + `wnt`
   resumption) is on the critical path for any longer-session future.

### Known limits of this analysis
Best-case eviction model (firing reliability at scale is assumed from the
small-trigger h04c measurements, not measured at lean-SI configs); 2.5-only
calibration (3.1 what-ifs ~3% optimistic); text-in only in the what-ifs
(audio/output adds a roughly arm-independent constant); single run per matrix
cell (no variance bars); the GoAway-truncated cells contribute only their
live spans.